In [ ]:
from skimage.transform import rotate
import numpy as np
from skimage import morphology 
from scipy.spatial import ConvexHull
from sklearn.cluster import KMeans
from scipy.spatial.distance import pdist
from skimage.color import rgb2hsv
from scipy.stats import circmean, circvar
from math import ceil, floor
from skimage.transform import resize
import cv2
from numpy import nan
from skimage.segmentation import slic

# A: Asymmetry
def cut_mask(mask):
    '''Cut empty space from mask array such that it has smallest possible dimensions.

    Args:
        mask (numpy.ndarray): mask to cut

    Returns:
        cut_mask_ (numpy.ndarray): cut mask
    '''
    rows = np.any(mask, axis=1)  # True for rows that have any non-zero
    cols = np.any(mask, axis=0)  # True for cols that have any non-zero

    row_min, row_max = np.where(rows)[0][[0, -1]]
    col_min, col_max = np.where(cols)[0][[0, -1]]

    return mask[row_min:row_max+1, col_min:col_max+1]

def midpointGroup9(image):
    '''Find midpoint of image array.'''
    row_mid = image.shape[0] / 2
    col_mid = image.shape[1] / 2
    return row_mid, col_mid

def asymmetry(mask):
    '''Calculate asymmetry score between 0 and 1 from vertical and horizontal axis
    on a binary mask, 0 being complete symmetry, 1 being complete asymmetry,
    i.e. no pixels overlapping when folding mask on x- and y-axis

    Args:
        mask (numpy.ndarray): input mask

    Returns:
        asymmetry_score (float): Float between 0 and 1 indicating level of asymmetry.
    '''

    row_mid, col_mid = midpointGroup9(mask)

    # Split mask into halves hortizontally and vertically
    upper_half = mask[:ceil(row_mid), :]
    lower_half = mask[floor(row_mid):, :]
    left_half = mask[:, :ceil(col_mid)]
    right_half = mask[:, floor(col_mid):]

    # Flip one half for each axis
    flipped_lower = np.flip(lower_half, axis=0)
    flipped_right = np.flip(right_half, axis=1)

    # Use logical xor to find pixels where only one half is present
    hori_xor_area = np.logical_xor(upper_half, flipped_lower)
    vert_xor_area = np.logical_xor(left_half, flipped_right)

    # Calculate asymmetry score
    asymmetry_score = (hori_xor_area.sum() + vert_xor_area.sum()) / (mask.sum() * 2)

    return round(asymmetry_score, 4)

def rotation_asymmetry(mask, n: int):

    """Rotate the mask n times and calculate asymmetry for each rotation.
    Returns a dictionary of asymmetry scores for each rotation angle."""
    
    asymmetry_scores = {}

    for i in range(n):

        degrees = 90 * i / n

        rotated_mask = rotate(mask, degrees, order=0, preserve_range=True)  # keep binary
        cutted_mask = cut_mask(rotated_mask.astype(bool))

        asymmetry_scores[degrees] = asymmetry(cutted_mask)

    return asymmetry_scores

def mean_asymmetry(mask, rotations = 30):

    """Compute mean asymmetry score by averaging rotation_asymmetry results.
    More reliable than single-direction asymmetry."""

    asymmetry_scores = rotation_asymmetry(mask, rotations)
    mean_score = round(np.mean(asymmetry_scores), 4)

    return mean_score

# B: Border

def get_compactness(mask):

    """ Measures how round vs irregular a lesion is.
    Formula: perimeter^2 / (4 * pi * area)
    Higher value = less compact."""

    area = np.sum(mask)
    struct_el = morphology.disk(3)
    mask_eroded = morphology.erosion(mask, struct_el)
    perimeter = np.sum(mask.astype(int) - mask_eroded.astype(int))
    return perimeter**2 / (4 * np.pi * area)

def convexity_score(mask):

    """Calculate convexity score between 0 and 1,
    with 0 indicating a smoother border and 1 a more crooked border."""

    # Get coordinates of all pixels in the lesion mask
    coords = np.column_stack(np.nonzero(mask))
    #requires at least 3 points to form a polygon
    if len(coords) < 3:
        return 1.0
    # Compute convex hull of lesion pixels
    hull = ConvexHull(coords)
    # Compute area of lesion mask
    lesion_area = np.count_nonzero(mask)
    # Compute area of convex hull
    convex_hull_area = hull.area
    # Compute convexity as ratio of lesion area to convex hull
    convexity = lesion_area / convex_hull_area
    return convexity

# C: Color

def get_com_col(cluster, centroids, threshold=0.08):
    labels = np.arange(0, len(np.unique(cluster.labels_)) + 1) # Create a list of 'bins' [0, 1, 2...] for each unique color cluster found
    (hist, _) = np.histogram(cluster.labels_, bins=labels) #Count how many pixels per cluster
    hist = hist.astype("float")/hist.sum() #convert to percentages
    # Convert centroids to a NumPy array for easy indexing
    centroids = np.array(centroids)
    # Use boolean indexing to select only clusters whose percentage exceeds threshold
    dominant_colors = centroids[hist > threshold]

    return dominant_colors

def get_multicolor_rate(im, mask, n):

    """Measure the maximum color difference inside a lesion using KMeans clustering.

    Args:
        image (numpy.ndarray): Original RGB image of the lesion.
        mask (numpy.ndarray): Binary mask of the lesion.
        n (int): Number of color clusters to use in KMeans.

    Returns:
        float: Maximum color difference between cluster centers.
               Higher value = more color variation in the lesion."""
    # mask = color.rgb2gray(mask)
    im = resize(im, (im.shape[0] // 4, im.shape[1] // 4), anti_aliasing=True) #reduces by 1/4
    mask = resize(mask, (mask.shape[0] // 4, mask.shape[1] // 4), anti_aliasing=False) > 0
    
    # If a pixel is part of the lesion, it adds that pixel's RGB color to a list. It multiplies by 256 to convert the color back to standard 0–255 values.
    col_list = (im[mask] * 256).reshape(-1, 3) #reshape it to (number_of_pixels, 3)
    if col_list.shape[0] == 0:
        return 0.0

    cluster = KMeans(n_clusters=n, n_init=10).fit(col_list) #this groups all pixels in col_list into n main colors (centroids).
    com_col_list = get_com_col(cluster, cluster.cluster_centers_) #This function filters out colors that only appear in a tiny percentage of the lesion
    
    # Compute maximum Euclidean distance between dominant colors
    return np.max(pdist(com_col_list, metric='euclidean')) #euclidean distance = np.sqrt((R1 - R2)**2 + (G1 - G2)**2 + (B1 - B2)**2)

def slic_segmentation(image, mask, n_segments = 50, compactness = 0.1):
    '''Get SLIC segments of a lesion.

Args:
    image (np.ndarray): image to segment
    mask (np.ndarray): lesion area (True = lesion)
    n_segments (int): number of segments (default 50)
    compactness (float): balance color vs position (default 0.1)

Returns:
    np.ndarray: segmented lesion labels
    '''
    slic_segments = slic(image,
                    n_segments = n_segments,
                    compactness = compactness,
                    sigma = 1,
                    mask = mask,
                    start_label = 1,
                    channel_axis = 2)

    return slic_segments

def get_hsv_means(image, slic_segments):
    '''Get mean HSV values for each segment in a SLIC segmented image.

    Args:
        image (numpy.ndarray): original image
        slic_segments (numpy.ndarray): SLIC segmentation

    Returns:
        hsv_means (list): HSV mean values for each segment.
    '''

    hsv_image = rgb2hsv(image)
    hsv_means = []
    for i in range(1, np.max(slic_segments)+1):

        mask = slic_segments == i

        #Get average HSV values from segment
        hue_mean = circmean(hsv_image[:, :, 0][mask], high=1, low=0)
        sat_mean = np.mean(hsv_image[:, :, 1][mask])
        val_mean = np.mean(hsv_image[:, :, 2][mask])
        hsv_means.append(np.array([hue_mean, sat_mean, val_mean]))

    return hsv_means

def hsv_var(image, slic_segments):
    '''Get variance of HSV means for each segment in
    SLIC segmentation in hue, saturation and value channels

    Args:
        image (numpy.ndarray): image to compute color variance for
        slic_segments (numpy.ndarray): array containing SLIC segmentation

    Returns:
        hue_var (float): variance in hue channel segment means
        sat_var (float): variance in saturation channel segment means
        val_var (float): variance in value channel segment means.
    '''

    # If there is only 1 slic segment, return (0, 0, 0) (If the lesion is too small for multiple segments, there is no variation)
    if len(np.unique(slic_segments)) <= 2: # Use 2 since slic_segments also has 0 marking for area outside mask
        return 0, 0, 0

    hsv_means = get_hsv_means(image, slic_segments) #get the list of average colors.
    hsv_means = np.array(hsv_means)
    
    # Calculate the mathematical 'variance' for each channel
    hue_var = circvar(hsv_means[:, 0], high=1, low=0) # Color variation
    sat_var = np.nanvar(hsv_means[:, 1])  # Vibrancy variation
    val_var = np.nanvar(hsv_means[:, 2]) # Brightness variation

    return hue_var, sat_var, val_var